# Lexicon maintenance - slug tier 2 qualifiers (phase 8)

Thin driver; the logic is in `lang_tools.lexicon.ingestion.slug_qualifiers` and
`lang_tools.llm.runner`.

Concept slugs that still collide after tiers 0-1 (preferred lemma, then lexfile)
get a qualifier taken from the concept's own English gloss (`hot-temperature`,
`hot-stolen-smuggled`). The hash already keeps ids unique; the qualifier only makes
them legible.

1. **Worklist** - `collision_groups`: every slug more than one concept shares.
2. **Sample**, then **full run** as one batch.
3. **Check** - `check_qualifiers` accepts a qualifier only when it is 1-3 words,
   every word comes from that concept's gloss, it is not just the base word, and no
   other concept in the group got the same one. That mechanical rule is the
   review (phase 8 Decision 3): it cannot put a meaning into an id that the gloss
   does not already state. Rows go to `data/curated/slug_qualifiers.jsonl`.
4. **Rebuild** - the build re-slugs; curated rows key on `concept_key`, so every
   other curated file survives the re-slug.

The 2026-10 run used `gpt-5-mini` at low reasoning; reasoning was 74% of its
output tokens and is not needed here. Re-measure before the next run.

**Paid-run rule.** Every LLM job here follows the same order: run a small live sample,
read its measured usage and the extrapolated cost, compare models if the choice is
open, and only then flip `RUN_PAID` and submit the full batch. Nothing in this notebook
spends money until that flag is set by hand.

In [ ]:
import collections

from llm_core.chat.config.openai import ChatOpenAIConfig

from lang_tools.lexicon.ingestion.slug_qualifiers import SLUG_QUALIFIERS_FILE
from lang_tools.lexicon.ingestion.slug_qualifiers import check_qualifiers
from lang_tools.lexicon.ingestion.slug_qualifiers import collision_groups
from lang_tools.lexicon.maintenance import curated_path
from lang_tools.lexicon.maintenance import write_jsonl
from lang_tools.llm.runner import collect_batch
from lang_tools.llm.runner import load_checkpoint
from lang_tools.llm.runner import prompt_sha
from lang_tools.llm.runner import read_records
from lang_tools.llm.runner import run_chain
from lang_tools.llm.runner import submit_batch
from lang_tools.llm.runner import summarize_usage
from lang_tools.llm.slug_qualifier import build_slug_qualifier_chain
from lang_tools.params.lang_tools_params import get_lang_tools_params
from lang_tools.params.load_env import load_env

load_env()
data_fol = get_lang_tools_params().paths.data_fol
work = data_fol / "_raw" / "lexicon" / "maintenance" / "slug_qualifiers"
curated = curated_path(data_fol, SLUG_QUALIFIERS_FILE)

MODEL = "gpt-6-luna"
JOB_TAG = {"job": "lang-tools-slug-qualifiers"}
#: Batch prices per 1M tokens for the chosen model (check the OpenAI pricing page
#: before a run; the standard live tier is double).
PRICES = {"input_per_m": 0.05, "cached_per_m": 0.005, "output_per_m": 0.25}
RUN_PAID = False  # flip by hand only after reading the sample's measured cost


class NoReasoningConfig(ChatOpenAIConfig):
    """ChatOpenAIConfig with reasoning off (llm-core has no kwargs passthrough)."""

    reasoning_effort: str = "none"


chain = build_slug_qualifier_chain(NoReasoningConfig(model=MODEL))

## Worklist

In [ ]:
groups = {group.slug: group for group in collision_groups(data_fol)}
items = [(slug, group.chain_input()) for slug, group in groups.items()]
len(groups), sum(len(g.concepts) for g in groups.values())

## Sample (live)

In [ ]:
sample_cp = work / "sample.jsonl"
run_chain(chain, items, checkpoint=sample_cp, limit=20)
totals = summarize_usage(read_records(sample_cp))
(
    totals,
    f"estimated full run at batch prices: ${totals.cost(**PRICES) * len(items) / max(totals.records, 1):.2f}",
)

## Full run (one batch)

In [ ]:
batch_cp = work / "run.jsonl"
if RUN_PAID:
    batch_id = submit_batch(
        chain,
        items,
        checkpoint=batch_cp,
        workdir=work,
        extra_body={"reasoning_effort": "none"},
        metadata=JOB_TAG,
    )
    print(batch_id)

In [ ]:
# collect_batch(chain, batch_id, checkpoint=batch_cp)

## Check and write the curated file

Groups that collide anew after a rebuild are worth a second pass; groups already
covered keep their rows (merge by `concept_key` before writing if you run a
partial job).

In [ ]:
sha = prompt_sha(chain)
rows = []
for slug, output in load_checkpoint(batch_cp, chain.output_model).items():
    rows += check_qualifiers(groups[slug], output, model=MODEL, prompt_sha=sha)
write_jsonl(rows, curated)
collections.Counter(row.status for row in rows)

## Rebuild

Re-run `02_transform` and the quality gate. The gate's slug section reports the
distinct-slug count; the remaining collisions keep their tier-1 slug.